# 🦠 COVID-19 Deaths Prediction Pipeline
### Démonstration Python — Réplication du pipeline Dataiku DSS

**Auteur :** Souheil DABBABI — Data Scientist  
**Plateforme originale :** Dataiku DSS  
**Objectif :** Prédire le nombre de décès COVID-19 via un modèle de régression

---

Ce notebook reproduit les étapes clés du pipeline Dataiku DSS en Python natif :

| Étape | Description |
|-------|-------------|
| 1 | 📥 Chargement et exploration des données |
| 2 | 🔗 Jointure des datasets par pays |
| 3 | 🧹 Nettoyage et préparation des features |
| 4 | 🔍 Filtrage sur les États-Unis |
| 5 | 🔀 Division Train / Test |
| 6 | 🤖 Entraînement du modèle de régression |
| 7 | 📊 Scoring et prédictions |
| 8 | 🏆 Évaluation des performances |

---
## ⚙️ 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2E75B6', '#1F4E79', '#BDD7EE', '#FF6B35', '#2AB1AC']

print('✅ Imports OK')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

---
## 📥 Étape 1 — Chargement et Exploration des Données
> *Dans Dataiku DSS : Import de `02_05_2021` et `Continent_Country_Mapping`*

In [ ]:
# ── Génération de données synthétiques représentatives du dataset COVID ──────
# (remplacer par pd.read_csv('02_05_2021.csv') avec le vrai dataset)

np.random.seed(42)
n = 5000

countries = [
    'US', 'US', 'US', 'US', 'Brazil', 'India', 'France',
    'Germany', 'Italy', 'Spain', 'UK', 'Mexico', 'Russia', 'Turkey'
]

country_col = np.random.choice(countries, n, p=[
    0.30, 0.00, 0.00, 0.00, 0.10, 0.10, 0.07,
    0.07, 0.07, 0.07, 0.07, 0.05, 0.05, 0.05
])

confirmed = np.random.randint(1000, 500000, n)
recovered = (confirmed * np.random.uniform(0.4, 0.9, n)).astype(int)
active    = confirmed - recovered - np.random.randint(0, 5000, n)
deaths    = (confirmed * np.random.uniform(0.005, 0.04, n)).astype(int)

df_covid = pd.DataFrame({
    'Country':   country_col,
    'Confirmed': confirmed,
    'Recovered': recovered,
    'Active':    active.clip(0),
    'Deaths':    deaths,
    'Date':      pd.date_range('2020-01-01', periods=n, freq='H').date
})

# ── Table de mapping pays → continent ────────────────────────────────────────
continent_map = pd.DataFrame({
    'Country':   ['US', 'Brazil', 'India', 'France', 'Germany',
                  'Italy', 'Spain', 'UK', 'Mexico', 'Russia', 'Turkey'],
    'Continent': ['North America', 'South America', 'Asia', 'Europe', 'Europe',
                  'Europe', 'Europe', 'Europe', 'North America', 'Europe', 'Asia']
})

print(f'📊 Dataset COVID   : {df_covid.shape[0]:,} lignes × {df_covid.shape[1]} colonnes')
print(f'🗺️  Dataset Mapping : {continent_map.shape[0]} pays × {continent_map.shape[1]} colonnes')
print()
print(df_covid.head())

In [ ]:
# Exploration rapide
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution des décès
axes[0].hist(df_covid['Deaths'], bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution des Décès COVID-19', fontsize=13, fontweight='bold', color='#1F4E79')
axes[0].set_xlabel('Nombre de décès')
axes[0].set_ylabel('Fréquence')

# Top pays par décès
top_countries = df_covid.groupby('Country')['Deaths'].sum().sort_values(ascending=False).head(8)
top_countries.plot(kind='bar', ax=axes[1], color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[1].set_title('Total Décès par Pays', fontsize=13, fontweight='bold', color='#1F4E79')
axes[1].set_xlabel('Pays')
axes[1].set_ylabel('Total décès')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('exploration.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Exploration OK')

---
## 🔗 Étape 2 — Jointure des Datasets
> *Dans Dataiku DSS : Recette Join → `Joined_covid_Data` (LEFT JOIN sur Country)*

In [ ]:
# LEFT JOIN sur la colonne Country
df_joined = pd.merge(
    df_covid,
    continent_map,
    on='Country',
    how='left'   # ← LEFT JOIN : conservation de tous les enregistrements COVID
)

print(f'✅ Jointure effectuée')
print(f'   Avant jointure  : {df_covid.shape}')
print(f'   Après jointure  : {df_joined.shape}')
print(f'   Colonnes ajoutées : {set(df_joined.columns) - set(df_covid.columns)}')
print()
print(df_joined.head(3))

---
## 🧹 Étape 3 — Préparation et Nettoyage des Données
> *Dans Dataiku DSS : Recette Prepare → `Joined_covid_Data_prepared`*

In [ ]:
df_prep = df_joined.copy()

# 1. Valeurs manquantes
print('🔍 Valeurs manquantes avant nettoyage :')
print(df_prep.isnull().sum())

df_prep['Continent'].fillna('Unknown', inplace=True)
df_prep['Active'] = df_prep['Active'].clip(lower=0)

# 2. Feature Engineering
df_prep['Mortality_Rate']  = df_prep['Deaths'] / (df_prep['Confirmed'] + 1)
df_prep['Recovery_Rate']   = df_prep['Recovered'] / (df_prep['Confirmed'] + 1)
df_prep['Active_Rate']     = df_prep['Active'] / (df_prep['Confirmed'] + 1)

# 3. Encodage des variables catégorielles
le_country   = LabelEncoder()
le_continent = LabelEncoder()
df_prep['Country_enc']   = le_country.fit_transform(df_prep['Country'])
df_prep['Continent_enc'] = le_continent.fit_transform(df_prep['Continent'])

# 4. Suppression des colonnes inutiles pour le modèle
df_prep.drop(columns=['Date'], inplace=True)

print()
print(f'✅ Préparation OK — Shape : {df_prep.shape}')
print(f'   Features créées : Mortality_Rate, Recovery_Rate, Active_Rate')
print(f'   Encodages : Country_enc, Continent_enc')

---
## 🔍 Étape 4 — Filtrage sur les États-Unis
> *Dans Dataiku DSS : Recette Filter → `Country == 'US'`*

In [ ]:
df_us = df_prep[df_prep['Country'] == 'US'].copy()

print(f'✅ Filtrage sur US')
print(f'   Avant filtre : {len(df_prep):,} lignes')
print(f'   Après filtre : {len(df_us):,} lignes US uniquement')
print()
print(df_us[['Confirmed','Recovered','Active','Deaths','Mortality_Rate']].describe().round(4))

---
## 🔀 Étape 5 — Division Train / Test
> *Dans Dataiku DSS : Recette Split → `Train_Dataset` (80%) + `Test` (20%)*

In [ ]:
# Features pour le modèle
FEATURES = ['Confirmed', 'Recovered', 'Active',
            'Mortality_Rate', 'Recovery_Rate', 'Active_Rate',
            'Country_enc', 'Continent_enc']
TARGET = 'Deaths'

X = df_us[FEATURES]
y = df_us[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'✅ Split Train / Test effectué')
print(f'   Train_Dataset : {X_train.shape[0]:,} lignes ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Test          : {X_test.shape[0]:,} lignes ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'   Features utilisées : {FEATURES}')
print(f'   Variable cible     : {TARGET}')

---
## 🤖 Étape 6 — Entraînement du Modèle (AutoML Regression)
> *Dans Dataiku DSS : Lab AutoML → Predict Deaths (regression)*

In [ ]:
# Comparaison de plusieurs algorithmes (comme Dataiku AutoML)
models = {
    'Ridge Regression':      Ridge(alpha=1.0),
    'Random Forest':         RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
trained_models = {}

print('🏋️  Entraînement en cours...\n')
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    results[name] = {
        'R² (train)':  round(r2_score(y_train, y_pred_train), 4),
        'R² (test)':   round(r2_score(y_test,  y_pred_test),  4),
        'RMSE':        round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 2),
        'MAE':         round(mean_absolute_error(y_test, y_pred_test), 2)
    }
    trained_models[name] = model
    print(f'  ✅ {name}')
    print(f'     R² test={results[name]["R² (test)"]:.4f} | RMSE={results[name]["RMSE"]:.1f} | MAE={results[name]["MAE"]:.1f}')

df_results = pd.DataFrame(results).T
print()
print('📊 Tableau comparatif des modèles :')
print(df_results.to_string())

# Sélection du meilleur modèle (R² test le plus élevé)
best_name  = df_results['R² (test)'].idxmax()
best_model = trained_models[best_name]
print(f'\n🏆 Meilleur modèle sélectionné : {best_name}')

---
## 📊 Étape 7 — Scoring sur le Dataset Test
> *Dans Dataiku DSS : Recette Score → `Test_scored`*

In [ ]:
# Génération des prédictions (Test_scored)
y_pred = best_model.predict(X_test)

test_scored = X_test.copy()
test_scored['Deaths_actual']    = y_test.values
test_scored['Deaths_predicted'] = y_pred.round().astype(int)
test_scored['Residual']         = test_scored['Deaths_actual'] - test_scored['Deaths_predicted']
test_scored['Abs_Error']        = test_scored['Residual'].abs()

print(f'✅ Dataset Test_scored généré : {test_scored.shape}')
print()
print(test_scored[['Deaths_actual','Deaths_predicted','Residual','Abs_Error']].head(10).to_string())

---
## 🏆 Étape 8 — Évaluation du Modèle
> *Dans Dataiku DSS : Evaluate Recipe — Métriques et visualisations*

In [ ]:
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

print('=' * 50)
print(f'  🏆  PERFORMANCE DU MODÈLE — {best_name}')
print('=' * 50)
print(f'  R²   : {r2:.4f}   (1.0 = parfait)')
print(f'  RMSE : {rmse:.2f}')
print(f'  MAE  : {mae:.2f}')
print('=' * 50)

# ── Dashboard d'évaluation ────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Predicted vs Actual
ax1 = fig.add_subplot(gs[0, :2])
ax1.scatter(y_test, y_pred, alpha=0.5, color=PALETTE[0], s=20, label='Prédictions')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax1.plot(lims, lims, 'r--', lw=1.5, label='Parfait (y=x)')
ax1.set_xlabel('Valeurs Réelles (Deaths)', fontsize=11)
ax1.set_ylabel('Valeurs Prédites (Deaths)', fontsize=11)
ax1.set_title(f'Prédictions vs Valeurs Réelles  |  R² = {r2:.4f}', fontsize=13, fontweight='bold', color='#1F4E79')
ax1.legend()

# 2. Metrics panel
ax2 = fig.add_subplot(gs[0, 2])
ax2.axis('off')
metrics_text = f"""
┌─────────────────────┐
│   MÉTRIQUES         │
├─────────────────────┤
│  R²   =  {r2:.4f}   │
│  RMSE =  {rmse:8.2f} │
│  MAE  =  {mae:8.2f} │
├─────────────────────┤
│  Modèle :           │
│  {best_name[:20]:<20} │
└─────────────────────┘
"""
ax2.text(0.05, 0.5, metrics_text, transform=ax2.transAxes,
         fontsize=11, verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor=PALETTE[2], alpha=0.4))

# 3. Résidus
residuals = y_test.values - y_pred
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(residuals, bins=40, color=PALETTE[3], edgecolor='white', alpha=0.8)
ax3.axvline(0, color='red', lw=1.5, linestyle='--')
ax3.set_title('Distribution des Résidus', fontsize=12, fontweight='bold', color='#1F4E79')
ax3.set_xlabel('Résidu')
ax3.set_ylabel('Fréquence')

# 4. Feature Importance (si Random Forest ou GBM)
ax4 = fig.add_subplot(gs[1, 1])
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
    importances.plot(kind='barh', ax=ax4, color=PALETTE[0], edgecolor='white')
    ax4.set_title('Feature Importance', fontsize=12, fontweight='bold', color='#1F4E79')
    ax4.set_xlabel('Importance')
else:
    ax4.text(0.5, 0.5, 'Feature importance\nnon disponible\npour ce modèle',
             ha='center', va='center', transform=ax4.transAxes, fontsize=11)
    ax4.axis('off')

# 5. Comparaison des modèles
ax5 = fig.add_subplot(gs[1, 2])
r2_scores = {k: v['R² (test)'] for k, v in results.items()}
bars = ax5.bar(range(len(r2_scores)), list(r2_scores.values()),
               color=[PALETTE[1] if k == best_name else PALETTE[2] for k in r2_scores],
               edgecolor='white')
ax5.set_xticks(range(len(r2_scores)))
ax5.set_xticklabels([k.replace(' ', '\n') for k in r2_scores.keys()], fontsize=9)
ax5.set_title('Comparaison R² (Test)', fontsize=12, fontweight='bold', color='#1F4E79')
ax5.set_ylabel('R²')
ax5.set_ylim(0, 1.05)
for bar, val in zip(bars, r2_scores.values()):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.suptitle('🏆 Tableau de Bord — Évaluation du Modèle COVID-19 Deaths Prediction',
             fontsize=14, fontweight='bold', color='#1F4E79', y=1.01)

plt.savefig('model_evaluation.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Évaluation terminée — graphiques sauvegardés')

---
## ✅ Résumé du Pipeline

| Étape | Outil Dataiku | Équivalent Python | Statut |
|-------|--------------|-------------------|--------|
| Import données | Dataset Import | `pd.read_csv()` | ✅ |
| Jointure | Join Recipe | `pd.merge(how='left')` | ✅ |
| Préparation | Prepare Recipe | `fillna`, `LabelEncoder` | ✅ |
| Filtrage | Filter Recipe | `df[df['Country']=='US']` | ✅ |
| Split | Split Recipe | `train_test_split` | ✅ |
| Entraînement | AutoML Lab | `sklearn` models | ✅ |
| Scoring | Score Recipe | `model.predict()` | ✅ |
| Évaluation | Evaluate Recipe | `r2_score`, `rmse`, `mae` | ✅ |

---

**Auteur :** Souheil DABBABI — Data Scientist  
**Pipeline original :** Dataiku DSS  
**GitHub :** [github.com/SouheilDABBABI](https://github.com/SouheilDABBABI)  
**LinkedIn :** [linkedin.com/in/souheil-dabbabi](https://linkedin.com/in/souheil-dabbabi)